# 第 1 天练习 —— 飞机性能对比助手

## 练习目标（理念）

做一个「两架飞机选哪架更适合洲际航线」的小工具：

- **输入**：两架飞机的型号/系列名（交互式 `input`）
- **过程**：拼出 Wikipedia URL → 抓取页面正文 → 交给 LLM 做对比分析
- **输出**：带推理与表格的 Markdown 选型报告

## 和本课 Day 1 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| requests + BeautifulSoup | `fetch_website_contents(url)` 抓 Wiki 正文 |
| system / user messages | 航空性能工程师人设 + 两篇正文拼进 user |
| Chat Completions API | `openai.chat.completions.create(...)` + `gpt-4o-mini` |
| Markdown 展示 | `display(Markdown(...))` |

## 怎么跑

1. 准备 `.env`：`OPENAI_API_KEY`（常见以 `sk-proj-` 开头）
2. 从上到下运行单元格；在输入框填两个型号（空格会被替换成 `_` 以匹配 Wiki 词条）
3. 最后一格调用 `review_aircraft_performance(...)` 生成对比报告


In [1]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 OPENAI_API_KEY
import os
# 从 dotenv 导入 load_dotenv：把 .env 文件里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 bs4 导入 BeautifulSoup：把 HTML 解析成可查询的 DOM 树
from bs4 import BeautifulSoup
# 导入标准库 requests：用 HTTP GET 抓取网页 HTML
import requests
# 从 IPython.display 导入展示工具：在笔记本里漂亮地显示 Markdown
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI 客户端类：调用云端 Chat Completions API
from openai import OpenAI


In [2]:
# ========== 抓取正文：浏览器 UA + 去噪 + 截断 2000 字符 ==========

# User-Agent：假装成常见桌面 Chrome，降低被网站直接拒绝的概率
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

def fetch_website_contents(url):
    """返回给定 url 处的网站标题和内容；截断为 2,000 个字符作为合理限制。"""
    # GET 目标 URL；headers 带上上面的 UA
    response = requests.get(url, headers=headers)
    # 用 html.parser 解析响应字节流为 soup
    soup = BeautifulSoup(response.content, "html.parser")
    # 取 <title>；没有标题就用占位英文串（影响模型输入，保留原文）
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        # 删掉脚本/样式/图片/输入框等噪声节点
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        # 整页正文：用换行分隔并 strip 空白
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        # 没有 <body> 时正文置空
        text = ""
    # 标题 + 正文，并截断到 2000 字符，控制后续 prompt 体积
    return (title + "\n\n" + text)[:2_000]


In [3]:
# ========== 环境：加载并校验 OpenAI API Key ==========

# override=True：用 .env 覆盖进程里已有的同名环境变量
load_dotenv(override=True)
# 从环境变量读取 OpenAI 密钥
api_key = os.getenv('OPENAI_API_KEY')

# 分支检查：缺密钥 / 前缀不对 / 首尾空白 —— 打印英文排错提示（保留原文）
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


In [ ]:
# ========== Prompt：航空性能工程师人设 + 选型任务说明（英文勿译）==========

# system prompt 保留英文：发给模型的指令，改译会改变回答风格/行为
system_prompt="""
You are an experienced Aircraft Performance Engineer, who has expertise in analyzing aircraft performance data.You are tasked with analyzing the key aircraft performance characteristics data and\
providing a report on which aircraft an airline should choose for thier intercontinental flights.
"""

# user prompt：后续会再拼接两篇 Wiki 正文；结构与拼写错误都保持原样
user_prompt="""
Here is the website content.Please provide your reccomendation on which aircraft an airline should choose for thier intercontinental flights with proper reasoning by considering \
    operating expenses and aircraft performance limitations. Illustrate the aircraft performance comparison in tabular format that would make the information more clear.
"""


In [ ]:
# ========== 核心函数：抓两篇 Wiki → 组 messages → 调 gpt-4o-mini ==========

def get_aircraft_performance_review(url1,url2):
    """从给定的两个 url 抓取飞机相关页面正文，并让模型给出选型对比报告。"""
    # 分别抓取两架飞机对应页面的标题+正文（已截断）
    contents1 = fetch_website_contents(url1)
    contents2 = fetch_website_contents(url2)
    # messages：system 定角色；user = 任务说明 + 两段正文拼接
    messages = [
       {"role": "system", "content": system_prompt},
       {"role": "user", "content": user_prompt + contents1 + contents2} 
    ]

    # 默认云端 OpenAI 客户端（密钥来自环境变量）
    openai = OpenAI()
    
    # 非流式一次拿完整回答；model id 保持 gpt-4o-mini
    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages
    )
    # 取出助手文本并去掉首尾空白后返回
    return response.choices[0].message.content.strip()


In [ ]:
# ========== 交互输入：型号名 → Wikipedia URL（空格改下划线）==========

# input 提示字符串保持英文（原作者文案，影响运行交互）
ac_name1 = input("Enter the aircraft1 family/model: ")
ac_name2 = input("Enter the aircraft2 family/model: ")
# Wiki 词条常用下划线代替空格，例如 Boeing 787 → Boeing_787
if " " in ac_name1:
    ac_name1 = ac_name1.replace(" ", "_")
if " " in ac_name2:
    ac_name2 = ac_name2.replace(" ", "_")
# 拼出英文维基百科词条 URL（域名与路径保留，改译会抓错站）
wiki_ac_name1_url = "https://en.wikipedia.org/wiki/" + ac_name1
wiki_ac_name2_url = "https://en.wikipedia.org/wiki/" + ac_name2

# 原 f-string 在花括号内带空格：{ac_name1 } —— 语法合法，保持原样勿「修正」
print(f"So, you want to compare {ac_name1 } Vs {ac_name2}.")


In [7]:
# ========== 包装展示：调用评审函数，并把 Markdown 渲染到笔记本 ==========

def review_aircraft_performance(url1,url2):
    # 注意：局部变量名与函数同名（原作者写法）；只在函数体内遮蔽外层名字
    review_aircraft_performance= get_aircraft_performance_review(url1,url2)
    # 用 Markdown 漂亮显示模型返回的对比报告
    display(Markdown(review_aircraft_performance))


In [8]:
# ========== 执行：用上一格拼好的两个 Wiki URL 跑完整对比流程 ==========

review_aircraft_performance(wiki_ac_name1_url,wiki_ac_name2_url)


To provide a recommendation for an airline considering aircraft options for intercontinental flights, I will analyze two popular aircraft models: the Boeing 737 MAX family and the Airbus A320neo family. Since specific performance data is missing from the provided input, I will rely on known characteristics of these aircraft as of October 2023.

### Boeing 737 MAX Overview
**Variants**: 
- 737 MAX 7
- 737 MAX 8
- 737 MAX 9
- 737 MAX 10

**Key Specifications**:
- **Range**: Approximately 3,550 to 4,700 nautical miles depending on the variant.
- **Passenger Capacity**: Ranges from about 138 to 230 passengers.
- **Cruise Speed**: Around Mach 0.79.
- **Fuel Efficiency**: The 737 MAX boasts a 14% lower fuel consumption compared to its predecessor (737 NG) and approximately 20% less fuel use than other aircraft in the same class.

**Strengths**:
- **Cost Efficiency**: Lower operating costs and fuel efficiency make it appealing to airlines.
- **Seat Capacity Flexibility**: Can serve various market demands due to different configurations.

**Weaknesses**:
- **Limited Range**: Though it can serve some transcontinental routes, many intercontinental routes (e.g., across the Atlantic or Pacific) may not be feasible without refueling.
  
### Airbus A320neo Overview
**Variants**:
- A319neo
- A320neo
- A321neo, including A321LR and A321XLR

**Key Specifications**:
- **Range**: The A321XLR variant has an impressive range of up to 4,700 nautical miles, making it suitable for longer-haul flights.
- **Passenger Capacity**: The A320neo family can accommodate between 140 to 244 passengers depending on the specific model and configuration.
- **Cruise Speed**: Similar to the 737 MAX at about Mach 0.78 to 0.80.
- **Fuel Efficiency**: Approximately 15% better fuel efficiency compared to the first generation A320 models, with ongoing improvements in aerodynamic design.

**Strengths**:
- **Longer Range**: The A321XLR can reach destinations that the 737 MAX family may not cover without fuel stops.
- **Greater Passenger Capacity**: The A321LR and A321XLR offer high capacity for more passengers on longer routes.

**Weaknesses**:
- **Higher Operating Costs**: Depending on configuration and flight duration, some operators may find the A320neo family exhibits higher operating costs compared to the smaller 737 MAX variants.

### Recommendation

For an airline looking to perform intercontinental flights efficiently, the **Airbus A321XLR stands out as the more suitable choice** due to its combination of long-range capability, flexible passenger configuration, and improved fuel efficiency compared to the Boeing 737 MAX series.

**Key Reasons**:
1. **Extended Range**: The ability to fly up to 4,700 nautical miles allows for non-stop intercontinental routes which is a critical factor for long haul operations.
2. **Seating Capacity**: The capacity to comfortably seat between 180 to 220 passengers (in varying configurations) helps to accommodate a larger number of travelers.
3. **Operational Flexibility**: The A321XLR allows airlines to target routes that are currently underserved or neglected due to traditional wide-body aircraft limitations, helping to tap into more profitable markets.
4. **Fuel Efficiency**: The latest advancements in the A320neo family, particularly the A321XLR, continue to improve operational costs, making it attractive in competitive intercontinental markets.

In conclusion, the Airbus A321XLR is the recommended aircraft for intercontinental services, providing airlines with the operational flexibility, range, and efficiency necessary for success in a competitive airline industry.